# Álgebra lineal para Machine Learning, escrita en NumPy

Casi todo lo que hace un modelo de aprendizaje automático se reduce a operaciones
sobre **vectores** y **matrices**: sumar, escalar, multiplicar, medir longitudes,
ángulos y distancias. Antes de ajustar un solo modelo conviene ver esa maquinaria
"desnuda", porque es exactamente la misma que reaparecerá en regresiones, redes
neuronales y algoritmos de agrupamiento.

En este cuaderno usamos el conjunto **Iris** (150 flores, 4 medidas cada una) como
excusa concreta para practicar:

| Concepto matemático | Traducción a NumPy | Para qué sirve en ML |
|---|---|---|
| Vector / matriz | `array`, `X` | Representar datos |
| Suma y escalar | `a + b`, `2 * a` | Combinar y reescalar |
| Producto punto | `np.dot`, `@` | Similitud, combinaciones lineales |
| Norma | `np.linalg.norm` | "Tamaño" de un vector |
| Coseno | fórmula | Parecido entre muestras |
| Distancia | `norm(u - v)` | Cercanía entre muestras |

> **Alcance:** aquí solo describimos los datos con álgebra lineal. Todavía **no**
> predecimos ni ajustamos ningún modelo — eso llega en la Unidad 3.

## 1. Los datos como una matriz

Un conjunto de datos tabular se organiza como una **matriz** $X$: cada **fila** es
una muestra (una flor) y cada **columna** es una característica (una medida).

$$
X \in \mathbb{R}^{n \times d},\qquad
X =
\begin{bmatrix}
x_{1,1} & x_{1,2} & \cdots & x_{1,d}\\
x_{2,1} & x_{2,2} & \cdots & x_{2,d}\\
\vdots  & \vdots  & \ddots & \vdots \\
x_{n,1} & x_{n,2} & \cdots & x_{n,d}
\end{bmatrix}
$$

Para Iris, $n = 150$ muestras y $d = 4$ características
(largo y ancho del sépalo, largo y ancho del pétalo).
El vector `y` guarda la etiqueta de especie de cada flor ($0,1,2$).

In [5]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()
X = iris.data      # matriz de características
y = iris.target    # etiquetas: 0, 1, 2
print(X.shape)
print(iris.feature_names)

(150, 4)
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


**Lectura del resultado.** `X.shape` es `(150, 4)`: confirma $n=150$ filas y
$d=4$ columnas. `iris.feature_names` da el significado de cada columna, en orden.
Que la forma sea `(150, 4)` — y no `(4, 150)` — fija la convención
**"muestras en las filas, características en las columnas"**, que seguiremos todo el cuaderno.

## 2. Una muestra es un vector

Tomar una fila de $X$ da un **vector** de $\mathbb{R}^d$: los $d$ números que
describen a esa flor.

$$
\mathbf{x}_0 = (x_{0,1},\, x_{0,2},\, x_{0,3},\, x_{0,4}) \in \mathbb{R}^4
$$

En NumPy, `X[0]` extrae la fila $0$. Su forma es `(4,)`: un arreglo de una sola
dimensión con 4 componentes.

In [8]:
x0 = X[0]          # primera muestra: un vector
print(x0)
print(x0.shape)

[5.1 3.5 1.4 0.2]
(4,)


**Lectura del resultado.** `[5.1 3.5 1.4 0.2]` son las cuatro medidas de la primera
flor, en el orden de `feature_names`. La forma `(4,)` (con la coma y sin segundo
número) indica un **vector 1-D**, no una matriz fila `(1, 4)`: esa distinción
importará cuando hagamos productos matriciales.

## 3. Una característica es una columna

Si en lugar de una fila tomamos una **columna**, obtenemos el valor de **una misma
característica para todas las muestras**:

$$
X_{:,\,j} = (x_{1,j},\, x_{2,j},\, \dots,\, x_{n,j}) \in \mathbb{R}^{n}
$$

La sintaxis `X[:, 2]` significa "todas las filas, columna 2" (la tercera
característica: largo del pétalo). El `:` es *slicing*: selecciona el rango completo
de ese eje.

In [14]:
# fila = muestra ; columna = característica
col_2 = X[:, 2]    # 3ª característica de TODAS las muestras
print(col_2[:5])
print(col_2.shape)

[1.4 1.4 1.3 1.5 1.4]
(150,)


**Lectura del resultado.** `col_2[:5]` muestra el largo de pétalo de las primeras 5
flores, y `col_2.shape` es `(150,)`: un vector con **una entrada por muestra**.
Filas y columnas son dos "cortes" distintos de la misma matriz — muestras frente a
características — y conviene tener siempre presente cuál se está tomando.

## 4. La transpuesta

**Transponer** una matriz intercambia sus filas y columnas:

$$
(X^{\top})_{ij} = X_{ji},\qquad
X \in \mathbb{R}^{n \times d}\ \Longrightarrow\ X^{\top} \in \mathbb{R}^{d \times n}
$$

Es una operación puramente de reordenamiento (no cambia los datos, solo su
disposición). Aparece constantemente en ML: por ejemplo, muchas fórmulas se
escriben como $X^{\top}X$.

In [15]:
print(X.shape)     # (150, 4)
print(X.T.shape)   # transpuesta: (4, 150)

(150, 4)
(4, 150)


**Lectura del resultado.** `X.shape` es `(150, 4)` y `X.T.shape` es `(4, 150)`:
la transpuesta convierte "150 flores × 4 medidas" en "4 medidas × 150 flores".
Ahora cada **fila** de $X^{\top}$ es una característica completa.

## 5. Operaciones vectoriales: suma y escalar

Los vectores se suman **componente a componente** y se escalan multiplicando cada
componente por un número $\alpha$:

$$
\mathbf{u} + \mathbf{v} = (u_1 + v_1,\ \dots,\ u_d + v_d),
\qquad
\alpha\,\mathbf{u} = (\alpha u_1,\ \dots,\ \alpha u_d)
$$

Estas dos operaciones son la base de toda **combinación lineal**
$\alpha\mathbf{u} + \beta\mathbf{v}$ — el ladrillo con el que se construyen
regresiones y capas de redes neuronales. NumPy las hace de forma *vectorizada*
(sin bucles) gracias al *broadcasting*.

In [17]:
a = X[0]           # una flor setosa
print(a)
b = X[50]          # una flor versicolor
print(b)
print('a + b =', a + b)
print('2 * a =', 2 * a)

[5.1 3.5 1.4 0.2]
[7.  3.2 4.7 1.4]
a + b = [12.1  6.7  6.1  1.6]
2 * a = [10.2  7.   2.8  0.4]


**Lectura del resultado.** `a + b` suma medida a medida las dos flores, y `2 * a`
duplica cada medida de la primera. Cada entrada del resultado depende **solo** de la
entrada correspondiente de las entradas: son operaciones que respetan la posición.

## 6. Producto punto (producto interno)

El **producto punto** combina dos vectores en **un solo número**:

$$
\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{d} u_i\, v_i
= u_1 v_1 + u_2 v_2 + \cdots + u_d v_d
$$

Es probablemente la operación más importante del ML: una predicción lineal
$\hat{y} = \mathbf{w}\cdot\mathbf{x}$, una proyección o una medida de similitud son,
en el fondo, productos punto. En NumPy hay tres formas equivalentes: a mano con un
bucle, `np.dot(u, v)` y el operador `@`.

In [12]:
manual = sum(ai * bi for ai, bi in zip(a, b))
print('a mano :', round(manual, 2))
print('np.dot :', round(np.dot(a, b), 2))
print('con @  :', round(a @ b, 2))

a mano : 53.76
np.dot : 53.76
con @  : 53.76


**Lectura del resultado.** Las tres vías dan **exactamente lo mismo** (`53.76`):
confirma que `np.dot` y `@` no son magia, solo la suma de productos de la fórmula,
pero calculada de forma optimizada. En la práctica se usa `@` o `np.dot`: son más
rápidos y legibles que el bucle.

## 7. Producto elemento a elemento (Hadamard)

Cuidado con no confundirlo con el producto punto. El **producto de Hadamard**
multiplica componente a componente y **devuelve un vector**, no un escalar:

$$
\mathbf{u} \odot \mathbf{v} = (u_1 v_1,\ u_2 v_2,\ \dots,\ u_d v_d)
$$

En NumPy, el operador `*` entre dos arreglos hace **esto** (Hadamard), no el
producto punto. El producto punto es, de hecho, "Hadamard y luego sumar":
$\mathbf{u}\cdot\mathbf{v} = \sum_i (\mathbf{u}\odot\mathbf{v})_i$.

In [18]:
a * b              # multiplicación elemento a elemento

array([35.7 , 11.2 ,  6.58,  0.28])

**Lectura del resultado.** `a * b` = `[35.7, 11.2, 6.58, 0.28]`: un vector de 4
componentes. Si sumas esas cuatro entradas obtienes `53.76`, el producto punto de la
celda anterior — una buena forma de recordar la relación entre ambas operaciones.

## 8. Norma y similitud coseno

La **norma** (euclidiana, o $L_2$) mide el "tamaño" o longitud de un vector, y se
define a partir del producto punto:

$$
\|\mathbf{u}\| = \sqrt{\sum_{i=1}^{d} u_i^2} = \sqrt{\mathbf{u}\cdot\mathbf{u}}
$$

La **similitud coseno** mide el **ángulo** entre dos vectores (no su tamaño):

$$
\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}
\in [-1,\ 1]
$$

Vale $1$ cuando apuntan en la misma dirección, $0$ cuando son perpendiculares y
$-1$ cuando son opuestos. Es una medida de **parecido de forma** insensible a la
escala, muy usada para comparar documentos, imágenes o, aquí, flores.

In [19]:
def coseno(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
 
print('cos(a, b) =', round(coseno(a, b), 4))

cos(a, b) = 0.9284


**Lectura del resultado.** `cos(a, b) = 0.9284`: cercano a $1$, así que la flor
setosa `a` y la versicolor `b` apuntan casi en la misma dirección (sus proporciones
se parecen), aunque no sean idénticas. La función reutiliza `np.dot` y
`np.linalg.norm`, es decir, es la fórmula anterior escrita tal cual.

## 9. Comparando flores con el coseno

Para que la medida "hable", conviene comparar pares con parentesco conocido: dos
flores de la **misma** especie frente a dos de **distinta** especie. Si el coseno
captura parecido, el primer par debería dar un valor más alto.

In [20]:
setosa1    = X[0]
setosa2    = X[1]
versicolor = X[50]
print('cos(setosa1, setosa2)    =', round(coseno(setosa1, setosa2), 4))
print('cos(setosa1, versicolor) =', round(coseno(setosa1, versicolor), 4))

cos(setosa1, setosa2)    = 0.9986
cos(setosa1, versicolor) = 0.9284


**Lectura del resultado.**

- `cos(setosa1, setosa2) = 0.9986` — dos setosas: casi $1$, muy parecidas.
- `cos(setosa1, versicolor) = 0.9284` — especies distintas: parecido menor.

El coseno **sí distingue** dentro-especie de entre-especies (0.9986 > 0.9284). Esa
capacidad de ordenar muestras por parecido es la semilla de algoritmos como *k-NN*
o el agrupamiento.

## 10. Distancia euclidiana

Mientras el coseno mira el ángulo, la **distancia euclidiana** mide qué tan
**lejos** están dos puntos, y es simplemente la norma de su diferencia:

$$
d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|
= \sqrt{\sum_{i=1}^{d} (u_i - v_i)^2}
$$

Aquí la escala **sí** importa: dos flores con medidas parecidas dan distancia
pequeña; con medidas muy diferentes, distancia grande.

In [21]:
def distancia(u, v):
    return np.linalg.norm(u - v)
 
print('d(setosa1, setosa2)    =', round(distancia(setosa1, setosa2), 4))
print('d(setosa1, versicolor) =', round(distancia(setosa1, versicolor), 4))

d(setosa1, setosa2)    = 0.5385
d(setosa1, versicolor) = 4.0037


**Lectura del resultado.**

- `d(setosa1, setosa2) = 0.5385` — misma especie: puntos **cercanos**.
- `d(setosa1, versicolor) = 4.0037` — especies distintas: puntos **lejanos**.

Coseno y distancia son dos lentes complementarias: el coseno compara **dirección**
(forma) y la distancia compara **posición** (magnitud). Muchos modelos eligen una u
otra según qué signifique "parecido" en el problema.

## 11. La relación $\|x\|^2 = x \cdot x$

Cerramos con una identidad que conecta todo lo anterior. Como la norma se define a
partir del producto punto, elevar la norma al cuadrado **deshace** la raíz:

$$
\mathbf{x}\cdot\mathbf{x} = \sum_{i=1}^{d} x_i^2 = \|\mathbf{x}\|^2
$$

No es una casualidad numérica: es la definición misma de norma. Esta igualdad
aparece por todas partes en ML — por ejemplo, el **error cuadrático** que minimizan
las regresiones es una norma al cuadrado, $\|\mathbf{y} - \hat{\mathbf{y}}\|^2$.

In [22]:
x = X[0]
print('x . x      =', round(np.dot(x, x), 4))
print('||x|| ** 2 =', round(np.linalg.norm(x) ** 2, 4))

x . x      = 40.26
||x|| ** 2 = 40.26


**Lectura del resultado.** `x . x` y `||x|| ** 2` dan el **mismo** valor (`40.26`),
comprobando la identidad de forma empírica. Que un producto punto de un vector
consigo mismo sea su longitud al cuadrado es el puente entre "álgebra"
(productos punto) y "geometría" (distancias y tamaños).

---

### Resumen

| Operación | Fórmula | Devuelve | Idea en ML |
|---|---|---|---|
| Producto punto | $\sum_i u_i v_i$ | escalar | predicción lineal, similitud |
| Hadamard | $(u_i v_i)_i$ | vector | operaciones elemento a elemento |
| Norma | $\sqrt{\mathbf{u}\cdot\mathbf{u}}$ | escalar | "tamaño" de un vector |
| Coseno | $\dfrac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$ | escalar $[-1,1]$ | parecido de forma |
| Distancia | $\|\mathbf{u}-\mathbf{v}\|$ | escalar $\ge 0$ | cercanía entre puntos |

Toda esta maquinaria — vectores, matrices, productos punto, normas — es la misma que
usaremos al ajustar modelos. Aquí solo la aplicamos a **describir** los datos.